In [0]:
bronze_df = spark.table("workspace.nyc_taxi.bronze_trips")

In [0]:
print("Rows before cleaning:", bronze_df.count())

Rows before cleaning: 9554778


In [0]:
silver_df = bronze_df

In [0]:
silver_df = silver_df.filter(
    silver_df.fare_amount >= 0
)

In [0]:
print(silver_df.count())

9418211


In [0]:
silver_df = silver_df.filter(
    silver_df.total_amount >= 0
)

In [0]:
silver_df = silver_df.filter(
    (silver_df.trip_distance >= 0) &
    (silver_df.trip_distance <= 100)
)

In [0]:
silver_df = silver_df.filter(
    (silver_df.trip_distance >= 0) &
    (silver_df.trip_distance <= 100)
)

In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
silver_df = silver_df.fillna({
    "Airport_fee": 0
})

In [0]:
silver_df = silver_df.fillna({
    "congestion_surcharge": 0
})

In [0]:
print("Bronze:", bronze_df.count())
print("Silver:", silver_df.count())

Bronze: 9554778
Silver: 9417627


In [0]:
from pyspark.sql.functions import col, sum

null_counts = silver_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
])

display(null_counts)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,0,0,730818,0,730818,730818,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nyc_taxi.silver_trips")